# IMC Prosperity Round 5 — Basket, Pair, and HMM Analysis

This notebook is built for the 50-product final round. It does three things:

1. maps the 10 product baskets,
2. finds pair / spread / basket opportunities,
3. fits a 3-state Gaussian Hidden Markov Model on rolling returns for each product.

The outputs are intended to feed directly into the final trader.

In [7]:
from pathlib import Path
import json
import math
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

def find_round5_data_dir():
    """Find the Round 5 data folder from the repo root or the round folder."""
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend([base / "data", base / "round 5" / "data"])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if list(candidate.glob("prices_round_5_day_*.csv")) and list(candidate.glob("trades_round_5_day_*.csv")):
            return candidate

    searched = "\n".join(str(p) for p in seen)
    raise FileNotFoundError(
        "Could not find Round 5 data files. Expected prices_round_5_day_*.csv "
        "and trades_round_5_day_*.csv under a data folder. Searched:\n"
        f"{searched}"
    )

DATA_DIR = find_round5_data_dir()

PRICE_FILES = sorted(DATA_DIR.glob("prices_round_5_day_*.csv"))
TRADE_FILES = sorted(DATA_DIR.glob("trades_round_5_day_*.csv"))

PRICE_FILES, TRADE_FILES

([WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/prices_round_5_day_2.csv'),
  WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/prices_round_5_day_3.csv'),
  WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/prices_round_5_day_4.csv')],
 [WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/trades_round_5_day_2.csv'),
  WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/trades_round_5_day_3.csv'),
  WindowsPath('C:/Users/amans/OneDrive/Documents/GitHub/imc_prosperity_4/round 5/data/trades_round_5_day_4.csv')])

## 1. Universe definition

The round has 10 baskets of 5 products each. The notebook keeps the groups explicit so that every analysis cell can be run at both product and basket level.

In [8]:
PRODUCT_GROUPS = {
    "GALAXY_SOUNDS": [
        "GALAXY_SOUNDS_DARK_MATTER",
        "GALAXY_SOUNDS_BLACK_HOLES",
        "GALAXY_SOUNDS_PLANETARY_RINGS",
        "GALAXY_SOUNDS_SOLAR_WINDS",
        "GALAXY_SOUNDS_SOLAR_FLAMES",
    ],
    "SLEEP_POD": [
        "SLEEP_POD_SUEDE",
        "SLEEP_POD_LAMB_WOOL",
        "SLEEP_POD_POLYESTER",
        "SLEEP_POD_NYLON",
        "SLEEP_POD_COTTON",
    ],
    "MICROCHIP": [
        "MICROCHIP_CIRCLE",
        "MICROCHIP_OVAL",
        "MICROCHIP_SQUARE",
        "MICROCHIP_RECTANGLE",
        "MICROCHIP_TRIANGLE",
    ],
    "PEBBLES": [
        "PEBBLES_XS",
        "PEBBLES_S",
        "PEBBLES_M",
        "PEBBLES_L",
        "PEBBLES_XL",
    ],
    "ROBOT": [
        "ROBOT_VACUUMING",
        "ROBOT_MOPPING",
        "ROBOT_DISHES",
        "ROBOT_LAUNDRY",
        "ROBOT_IRONING",
    ],
    "UV_VISOR": [
        "UV_VISOR_YELLOW",
        "UV_VISOR_AMBER",
        "UV_VISOR_ORANGE",
        "UV_VISOR_RED",
        "UV_VISOR_MAGENTA",
    ],
    "TRANSLATOR": [
        "TRANSLATOR_SPACE_GRAY",
        "TRANSLATOR_ASTRO_BLACK",
        "TRANSLATOR_ECLIPSE_CHARCOAL",
        "TRANSLATOR_GRAPHITE_MIST",
        "TRANSLATOR_VOID_BLUE",
    ],
    "PANEL": [
        "PANEL_1X2",
        "PANEL_2X2",
        "PANEL_1X4",
        "PANEL_2X4",
        "PANEL_4X4",
    ],
    "OXYGEN_SHAKE": [
        "OXYGEN_SHAKE_MORNING_BREATH",
        "OXYGEN_SHAKE_EVENING_BREATH",
        "OXYGEN_SHAKE_MINT",
        "OXYGEN_SHAKE_CHOCOLATE",
        "OXYGEN_SHAKE_GARLIC",
    ],
    "SNACKPACK": [
        "SNACKPACK_CHOCOLATE",
        "SNACKPACK_VANILLA",
        "SNACKPACK_PISTACHIO",
        "SNACKPACK_STRAWBERRY",
        "SNACKPACK_RASPBERRY",
    ],
}
ALL_PRODUCTS = [p for g in PRODUCT_GROUPS.values() for p in g]
PRODUCT_TO_GROUP = {p: g for g, ps in PRODUCT_GROUPS.items() for p in ps}

len(ALL_PRODUCTS), len(PRODUCT_GROUPS)

(50, 10)

## 2. Data loading and standardization

The raw price files contain full order-book snapshots. The trade files contain aggressive executions and allow fill-pressure analysis later.

In [9]:
def load_csv_semicolon(path):
    return pd.read_csv(path, sep=";")

# Re-resolve these here so this cell also works after a cwd/kernel change.
DATA_DIR = find_round5_data_dir()
PRICE_FILES = sorted(DATA_DIR.glob("prices_round_5_day_*.csv"))
TRADE_FILES = sorted(DATA_DIR.glob("trades_round_5_day_*.csv"))

if not PRICE_FILES or not TRADE_FILES:
    raise FileNotFoundError(f"No Round 5 CSV files found in {DATA_DIR.resolve()}")

price_dfs = [load_csv_semicolon(p) for p in PRICE_FILES]
trade_dfs = [load_csv_semicolon(p) for p in TRADE_FILES]

prices = pd.concat(price_dfs, ignore_index=True)
trades = pd.concat(trade_dfs, ignore_index=True)

prices.columns = [c.strip().lower() for c in prices.columns]
trades.columns = [c.strip().lower() for c in trades.columns]

prices.head(), trades.head()

(   day  timestamp                    product  bid_price_1  bid_volume_1  bid_price_2  bid_volume_2  bid_price_3  bid_volume_3  ask_price_1  ask_volume_1  ask_price_2  ask_volume_2  ask_price_3  \
 0    2          0                  PEBBLES_L         9994            13       9992.0          21.0          NaN           NaN        10006            13      10008.0          21.0          NaN   
 1    2          0        SNACKPACK_RASPBERRY         9992            36       9990.0          45.0          NaN           NaN        10008            36      10010.0          45.0          NaN   
 2    2          0               UV_VISOR_RED         9994            22       9992.0          26.0          NaN           NaN        10006            22      10008.0          26.0          NaN   
 3    2          0                  PEBBLES_M         9994            13       9992.0          21.0          NaN           NaN        10006            13      10008.0          21.0          NaN   
 4    2        

In [10]:
def pick_col(df, candidates):
    cols = list(df.columns)
    for cand in candidates:
        if cand in cols:
            return cand
    for cand in candidates:
        for c in cols:
            if cand in c:
                return c
    return None

PRICE_COLS = {
    "product": pick_col(prices, ["product", "symbol", "name"]),
    "timestamp": pick_col(prices, ["timestamp", "time"]),
    "day": pick_col(prices, ["day"]),
    "mid": pick_col(prices, ["mid_price", "mid"]),
    "bid1": pick_col(prices, ["bid_price_1"]),
    "ask1": pick_col(prices, ["ask_price_1"]),
    "bidv1": pick_col(prices, ["bid_volume_1"]),
    "askv1": pick_col(prices, ["ask_volume_1"]),
}

TRADE_COLS = {
    "product": pick_col(trades, ["symbol", "product", "name"]),
    "timestamp": pick_col(trades, ["timestamp", "time"]),
    "price": pick_col(trades, ["price"]),
    "qty": pick_col(trades, ["quantity", "qty", "volume"]),
    "buyer": pick_col(trades, ["buyer"]),
    "seller": pick_col(trades, ["seller"]),
}

PRICE_COLS, TRADE_COLS

({'product': 'product',
  'timestamp': 'timestamp',
  'day': 'day',
  'mid': 'mid_price',
  'bid1': 'bid_price_1',
  'ask1': 'ask_price_1',
  'bidv1': 'bid_volume_1',
  'askv1': 'ask_volume_1'},
 {'product': 'symbol',
  'timestamp': 'timestamp',
  'price': 'price',
  'qty': 'quantity',
  'buyer': 'buyer',
  'seller': 'seller'})

In [11]:
def add_features(df):
    df = df.copy()
    df["mid"] = pd.to_numeric(df[PRICE_COLS["mid"]], errors="coerce")
    if PRICE_COLS["bid1"] and PRICE_COLS["ask1"]:
        df["spread"] = pd.to_numeric(df[PRICE_COLS["ask1"]], errors="coerce") - pd.to_numeric(df[PRICE_COLS["bid1"]], errors="coerce")
        if PRICE_COLS["bidv1"] and PRICE_COLS["askv1"]:
            bidv = pd.to_numeric(df[PRICE_COLS["bidv1"]], errors="coerce").abs()
            askv = pd.to_numeric(df[PRICE_COLS["askv1"]], errors="coerce").abs()
            denom = (bidv + askv).replace(0, np.nan)
            df["obi"] = (bidv - askv) / denom
            df["microprice"] = (
                pd.to_numeric(df[PRICE_COLS["bid1"]], errors="coerce") * askv
                + pd.to_numeric(df[PRICE_COLS["ask1"]], errors="coerce") * bidv
            ) / denom
        else:
            df["obi"] = np.nan
            df["microprice"] = np.nan
    else:
        df["spread"] = np.nan
        df["obi"] = np.nan
        df["microprice"] = np.nan

    df["ret"] = df.groupby(PRICE_COLS["product"])["mid"].transform(lambda s: np.log(s).diff())
    return df

prices_f = add_features(prices)
prices_f.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,mid,spread,obi,microprice,ret
0,2,0,PEBBLES_L,9994,13,9992.0,21.0,NaN,NaN,10006,13,10008.0,21.0,NaN,NaN,10000.0,0.0,10000.0,12,0.0,10000.0,NaN
1,2,0,SNACKPACK_RASPBERRY,9992,36,9990.0,45.0,NaN,NaN,10008,36,10010.0,45.0,NaN,NaN,10000.0,0.0,10000.0,16,0.0,10000.0,NaN
2,2,0,UV_VISOR_RED,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,10000.0,12,0.0,10000.0,NaN
3,2,0,PEBBLES_M,9994,13,9992.0,21.0,NaN,NaN,10006,13,10008.0,21.0,NaN,NaN,10000.0,0.0,10000.0,12,0.0,10000.0,NaN
4,2,0,GALAXY_SOUNDS_DARK_MATTER,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,10000.0,12,0.0,10000.0,NaN


## 3. Product-level dashboard

This table is the first screen I would use to decide whether a product belongs to a trend bucket, a market-making bucket, or a pair-trading bucket.

In [12]:
def product_summary(df):
    out = []
    for product, g in df.groupby(PRICE_COLS["product"]):
        g = g.sort_values([PRICE_COLS["day"], PRICE_COLS["timestamp"]])
        mid = g["mid"].dropna().astype(float)
        ret = g["ret"].dropna().astype(float)
        spread = g["spread"].dropna().astype(float)
        if len(mid) < 20:
            continue
        slope = np.polyfit(np.arange(len(mid)), mid.values, 1)[0] if len(mid) > 1 else np.nan
        out.append({
            "product": product,
            "n": len(g),
            "mean_mid": mid.mean(),
            "std_mid": mid.std(),
            "cv_mid": mid.std() / mid.mean() if mid.mean() else np.nan,
            "slope": slope,
            "ret_mean": ret.mean() if len(ret) else np.nan,
            "ret_std": ret.std() if len(ret) else np.nan,
            "spread_mean": spread.mean() if len(spread) else np.nan,
            "spread_median": spread.median() if len(spread) else np.nan,
            "autocorr_mid_1": mid.autocorr(1),
        })
    return pd.DataFrame(out).sort_values("slope")

prod_summary = product_summary(prices_f)
display(prod_summary.head(20))

,product,n,mean_mid,std_mid,cv_mid,slope,ret_mean,ret_std,spread_mean,spread_median,autocorr_mid_1
6,MICROCHIP_OVAL,30000,8179.598717,1551.911621,0.189730,-0.171087,-1.981361e-05,0.001499,7.449767,8.0,0.999968
24,PEBBLES_XS,30000,7404.639767,1449.546839,0.195762,-0.158819,-1.681764e-05,0.002141,9.744867,9.0,0.999946
45,UV_VISOR_AMBER,30000,7911.696250,996.917584,0.126006,-0.109939,-1.127617e-05,0.001006,10.320500,10.0,0.999968
22,PEBBLES_S,30000,8932.356733,833.282141,0.093288,-0.085975,-7.162419e-06,0.001705,11.551533,12.0,0.999838
26,ROBOT_IRONING,30000,8701.570267,771.029728,0.088608,-0.077917,-8.154358e-06,0.001177,6.392933,6.0,0.999908
9,MICROCHIP_TRIANGLE,30000,9686.391083,833.369596,0.086035,-0.077144,-7.683020e-06,0.001491,8.635433,9.0,0.999849
16,PANEL_1X4,30000,9397.580617,834.033483,0.088750,-0.070208,-2.679987e-06,0.001005,8.376633,8.0,0.999935
7,MICROCHIP_RECTANGLE,30000,8732.439350,752.019290,0.086118,-0.065612,-4.367488e-06,0.001499,7.885767,8.0,0.999848
27,ROBOT_LAUNDRY,30000,9822.761700,614.322320,0.062541,-0.056277,-2.582592e-06,0.000998,7.165067,7.0,0.999872
14,OXYGEN_SHAKE_MORNING_BREATH,30000,10000.453350,652.805090,0.065278,-0.056197,-1.533104e-06,0.001007,12.782900,13.0,0.999880


## 4. Basket-level aggregation

For each 5-name basket we compute a simple synthetic average, then inspect deviations of individual names from that basket mean. This helps identify:
- a dominant basket anchor,
- stable offsets inside the basket,
- names that are rich or cheap versus their family.

In [13]:
def basket_table(df, group_name):
    group = PRODUCT_GROUPS[group_name]
    g = df[df[PRICE_COLS["product"]].isin(group)].copy()
    piv = g.pivot_table(index=[PRICE_COLS["day"], PRICE_COLS["timestamp"]], columns=PRICE_COLS["product"], values="mid", aggfunc="last").sort_index()
    piv["basket_mean"] = piv[group].mean(axis=1)
    for p in group:
        piv[f"resid_{p}"] = piv[p] - piv["basket_mean"]
    return piv

basket_tables = {g: basket_table(prices_f, g) for g in PRODUCT_GROUPS}
sample_group = "SNACKPACK"
display(basket_tables[sample_group].head())

product        SNACKPACK_CHOCOLATE  SNACKPACK_PISTACHIO  SNACKPACK_RASPBERRY  SNACKPACK_STRAWBERRY  SNACKPACK_VANILLA  basket_mean  resid_SNACKPACK_CHOCOLATE  resid_SNACKPACK_VANILLA  \
day timestamp                                                                                                                                                                            
2   0                      10000.0              10000.0              10000.0               10000.0            10000.0      10000.0                        0.0                      0.0   
    100                    10008.5               9997.5              10005.5                9994.5             9993.5       9999.9                        8.6                     -6.4   
    200                    10007.5              10000.5              10001.5                9999.5             9994.5      10000.7                        6.8                     -6.2   
    300                    10011.5               9999.5              10008.5                9991.5             9988.5       9999.9                       11.6                    -11.4   
    400                    10012.5               9988.5              10018.5                9980.5             9990.5       9998.1                       14.4                     -7.6   

product        resid_SNACKPACK_PISTACHIO  resid_SNACKPACK_STRAWBERRY  resid_SNACKPACK_RASPBERRY  
day timestamp                                                                                    
2   0                                0.0                         0.0                        0.0  
    100                             -2.4                        -5.4                        5.6  
    200                             -0.2                        -1.2                        0.8  
    300                             -0.4                        -8.4                        8.6  
    400                             -9.6                       -17.6                       20.4

In [14]:
basket_resid_summary = []
for g, piv in basket_tables.items():
    group = PRODUCT_GROUPS[g]
    row = {"group": g, "rows": len(piv)}
    for p in group:
        resid = piv[f"resid_{p}"].dropna()
        row[f"{p}_resid_mean"] = resid.mean()
        row[f"{p}_resid_std"] = resid.std()
    basket_resid_summary.append(row)

basket_resid_summary = pd.DataFrame(basket_resid_summary)
display(basket_resid_summary.head(10))

,group,rows,GALAXY_SOUNDS_DARK_MATTER_resid_mean,GALAXY_SOUNDS_DARK_MATTER_resid_std,GALAXY_SOUNDS_BLACK_HOLES_resid_mean,GALAXY_SOUNDS_BLACK_HOLES_resid_std,GALAXY_SOUNDS_PLANETARY_RINGS_resid_mean,GALAXY_SOUNDS_PLANETARY_RINGS_resid_std,GALAXY_SOUNDS_SOLAR_WINDS_resid_mean,GALAXY_SOUNDS_SOLAR_WINDS_resid_std,GALAXY_SOUNDS_SOLAR_FLAMES_resid_mean,GALAXY_SOUNDS_SOLAR_FLAMES_resid_std,SLEEP_POD_SUEDE_resid_mean,SLEEP_POD_SUEDE_resid_std,SLEEP_POD_LAMB_WOOL_resid_mean,SLEEP_POD_LAMB_WOOL_resid_std,SLEEP_POD_POLYESTER_resid_mean,SLEEP_POD_POLYESTER_resid_std,SLEEP_POD_NYLON_resid_mean,SLEEP_POD_NYLON_resid_std,SLEEP_POD_COTTON_resid_mean,SLEEP_POD_COTTON_resid_std,MICROCHIP_CIRCLE_resid_mean,MICROCHIP_CIRCLE_resid_std,MICROCHIP_OVAL_resid_mean,MICROCHIP_OVAL_resid_std,MICROCHIP_SQUARE_resid_mean,MICROCHIP_SQUARE_resid_std,MICROCHIP_RECTANGLE_resid_mean,MICROCHIP_RECTANGLE_resid_std,MICROCHIP_TRIANGLE_resid_mean,MICROCHIP_TRIANGLE_resid_std,PEBBLES_XS_resid_mean,PEBBLES_XS_resid_std,PEBBLES_S_resid_mean,PEBBLES_S_resid_std,PEBBLES_M_resid_mean,PEBBLES_M_resid_std,PEBBLES_L_resid_mean,PEBBLES_L_resid_std,PEBBLES_XL_resid_mean,PEBBLES_XL_resid_std,ROBOT_VACUUMING_resid_mean,ROBOT_VACUUMING_resid_std,ROBOT_MOPPING_resid_mean,ROBOT_MOPPING_resid_std,ROBOT_DISHES_resid_mean,ROBOT_DISHES_resid_std,ROBOT_LAUNDRY_resid_mean,ROBOT_LAUNDRY_resid_std,ROBOT_IRONING_resid_mean,ROBOT_IRONING_resid_std,UV_VISOR_YELLOW_resid_mean,UV_VISOR_YELLOW_resid_std,UV_VISOR_AMBER_resid_mean,UV_VISOR_AMBER_resid_std,UV_VISOR_ORANGE_resid_mean,UV_VISOR_ORANGE_resid_std,UV_VISOR_RED_resid_mean,UV_VISOR_RED_resid_std,UV_VISOR_MAGENTA_resid_mean,UV_VISOR_MAGENTA_resid_std,TRANSLATOR_SPACE_GRAY_resid_mean,TRANSLATOR_SPACE_GRAY_resid_std,TRANSLATOR_ASTRO_BLACK_resid_mean,TRANSLATOR_ASTRO_BLACK_resid_std,TRANSLATOR_ECLIPSE_CHARCOAL_resid_mean,TRANSLATOR_ECLIPSE_CHARCOAL_resid_std,TRANSLATOR_GRAPHITE_MIST_resid_mean,TRANSLATOR_GRAPHITE_MIST_resid_std,TRANSLATOR_VOID_BLUE_resid_mean,TRANSLATOR_VOID_BLUE_resid_std,PANEL_1X2_resid_mean,PANEL_1X2_resid_std,PANEL_2X2_resid_mean,PANEL_2X2_resid_std,PANEL_1X4_resid_mean,PANEL_1X4_resid_std,PANEL_2X4_resid_mean,PANEL_2X4_resid_std,PANEL_4X4_resid_mean,PANEL_4X4_resid_std,OXYGEN_SHAKE_MORNING_BREATH_resid_mean,OXYGEN_SHAKE_MORNING_BREATH_resid_std,OXYGEN_SHAKE_EVENING_BREATH_resid_mean,OXYGEN_SHAKE_EVENING_BREATH_resid_std,OXYGEN_SHAKE_MINT_resid_mean,OXYGEN_SHAKE_MINT_resid_std,OXYGEN_SHAKE_CHOCOLATE_resid_mean,OXYGEN_SHAKE_CHOCOLATE_resid_std,OXYGEN_SHAKE_GARLIC_resid_mean,OXYGEN_SHAKE_GARLIC_resid_std,SNACKPACK_CHOCOLATE_resid_mean,SNACKPACK_CHOCOLATE_resid_std,SNACKPACK_VANILLA_resid_mean,SNACKPACK_VANILLA_resid_std,SNACKPACK_PISTACHIO_resid_mean,SNACKPACK_PISTACHIO_resid_std,SNACKPACK_STRAWBERRY_resid_mean,SNACKPACK_STRAWBERRY_resid_std,SNACKPACK_RASPBERRY_resid_mean,SNACKPACK_RASPBERRY_resid_std
0,GALAXY_SOUNDS,30000,-571.402733,365.188725,668.807533,749.863651,-31.391367,617.319585,-360.520583,454.592387,294.50715,534.130262,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SLEEP_POD,30000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,376.718507,509.581211,-319.26021,626.768009,819.859023,488.020283,-1384.22936,521.600758,506.91204,435.764619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MICROCHIP,30000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-666.727263,719.325449,-1702.01383,1322.515015,3713.135753,1930.9

## 5. Cross-product correlation matrix and pair candidates

This is the pairs-trading screen. I compute the full correlation matrix on mid prices, then score candidate pairs by:
- high absolute correlation,
- stable hedge ratio,
- stationary-looking residual spread,
- usable spread volatility.

In [15]:
piv_all = prices_f.pivot_table(
    index=[PRICE_COLS["day"], PRICE_COLS["timestamp"]],
    columns=PRICE_COLS["product"],
    values="mid",
    aggfunc="last"
).sort_index()

corr = piv_all.corr()

pairs = []
cols = list(corr.columns)
for i, a in enumerate(cols):
    for b in cols[i+1:]:
        c = corr.loc[a, b]
        if pd.isna(c):
            continue
        s = piv_all[[a, b]].dropna()
        if len(s) < 100:
            continue
        y = s[b].values
        x = s[a].values
        beta = np.cov(x, y, ddof=1)[0, 1] / np.var(y, ddof=1) if np.var(y, ddof=1) > 0 else np.nan
        alpha = x.mean() - beta * y.mean() if np.isfinite(beta) else np.nan
        spread = x - (alpha + beta * y)
        spread_std = spread.std()
        spread_ac1 = pd.Series(spread).autocorr(1)
        pairs.append({
            "a": a,
            "b": b,
            "corr": c,
            "abs_corr": abs(c),
            "beta": beta,
            "alpha": alpha,
            "spread_std": spread_std,
            "spread_ac1": spread_ac1,
            "score": abs(c) / (spread_std + 1e-9),
        })

pairs_df = pd.DataFrame(pairs).sort_values(["abs_corr", "score"], ascending=False)
display(pairs_df.head(20))

,a,b,corr,abs_corr,beta,alpha,spread_std,spread_ac1,score
920,PEBBLES_XS,UV_VISOR_AMBER,0.958298,0.958298,1.393393,-3619.460044,414.231745,0.998981,0.002313
1100,SLEEP_POD_POLYESTER,UV_VISOR_AMBER,-0.940922,0.940922,-0.922632,19140.148571,331.012092,0.999101,0.002843
1123,SNACKPACK_CHOCOLATE,SNACKPACK_VANILLA,-0.925873,0.925873,-1.041111,20355.785260,75.842806,0.999345,0.012208
389,MICROCHIP_SQUARE,SLEEP_POD_SUEDE,0.918489,0.918489,1.867963,-7695.210416,723.754817,0.999155,0.001269
400,MICROCHIP_SQUARE,UV_VISOR_AMBER,-0.914151,0.914151,-1.678301,26872.953646,741.922925,0.999445,0.001232
379,MICROCHIP_SQUARE,PEBBLES_XS,-0.913768,0.913768,-1.153758,22137.909812,743.502566,0.999334,0.001229
1165,SNACKPACK_STRAWBERRY,UV_VISOR_AMBER,-0.893629,0.893629,-0.325904,13285.061238,163.171503,0.998619,0.005477
908,PEBBLES_XS,SLEEP_POD_POLYESTER,-0.893290,0.893290,-1.324617,23088.846290,651.533295,0.999439,0.001371
1115,SLEEP_POD_SUEDE,UV_VISOR_AMBER,-0.890926,0.890926,-0.804265,17760.521155,408.702035,0.999479,0.002180
11,GALAXY_SOUNDS_BLACK_HOLES,OXYGEN_SHAKE_GARLIC,0.885327,0.885327,0.890059,852.346548,445.639237,0.999390,0.001987


## 6. Trade-tape analysis

This section uses the trades file to infer:
- which products are more aggressively traded,
- average trade price versus mid,
- whether aggressive flow tends to precede price continuation or reversal.

In [ ]:
def trade_tape_summary(trades_df, prices_df):
    t = trades_df.copy()
    p = prices_df.copy()

    t_prod = TRADE_COLS["product"]
    t_ts = TRADE_COLS["timestamp"]
    t_price = TRADE_COLS["price"]
    t_qty = TRADE_COLS["qty"]
    t_buyer = TRADE_COLS["buyer"]
    t_seller = TRADE_COLS["seller"]

    p_prod = PRICE_COLS["product"]
    p_ts = PRICE_COLS["timestamp"]

    rows = []
    for product, g in t.groupby(t_prod):
        g = g.sort_values(t_ts)
        psub = p[p[p_prod] == product].sort_values(p_ts)[[p_ts, "mid"]].dropna()
        if len(g) < 5 or len(psub) < 5:
            continue

        merged = pd.merge_asof(
            g[[t_ts, t_price, t_qty]].rename(columns={t_ts: "timestamp"}),
            psub.rename(columns={p_ts: "timestamp"}),
            on="timestamp",
            direction="backward",
        ).dropna()

        buyer_known = (g[t_buyer].astype(str) == "SUBMISSION").sum() if t_buyer else np.nan
        seller_known = (g[t_seller].astype(str) == "SUBMISSION").sum() if t_seller else np.nan

        rows.append({
            "product": product,
            "trades": len(g),
            "avg_trade_price": pd.to_numeric(g[t_price], errors="coerce").mean(),
            "avg_mid_at_trade": merged["mid"].mean(),
            "avg_signed_gap": (merged[t_price] - merged["mid"]).mean(),
            "qty_sum": pd.to_numeric(g[t_qty], errors="coerce").abs().sum(),
            "buyer_submission": buyer_known,
            "seller_submission": seller_known,
        })
    return pd.DataFrame(rows).sort_values("qty_sum", ascending=False)

tape_summary = trade_tape_summary(trades, prices_f)
display(tape_summary.head(20))

## 7. Hidden Markov Model: theory and implementation

This follows the standard Gaussian HMM with Baum-Welch training:
- forward-backward for posterior probabilities,
- log-space computation for stability,
- Viterbi for the most likely latent path,
- filtering for live trading and smoothing for historical analysis.

The regime model is the key mathematical bridge between raw price series and actual trading decisions. A causal filter is used in the live trader, while smoothing remains a notebook-only diagnostic. That distinction is standard HMM practice. fileciteturn5file0

In [17]:
def logsumexp(a, axis=None):
    a = np.asarray(a)
    amax = np.max(a, axis=axis, keepdims=True)
    out = amax + np.log(np.sum(np.exp(a - amax), axis=axis, keepdims=True))
    if axis is not None:
        out = np.squeeze(out, axis=axis)
    return out

def gaussian_logpdf(x, mu, var):
    var = np.maximum(var, 1e-12)
    return -0.5 * (np.log(2.0 * np.pi * var) + (x - mu) ** 2 / var)

def normalize_rows(a):
    a = np.asarray(a, dtype=float)
    a = np.clip(a, 1e-12, None)
    return a / a.sum(axis=1, keepdims=True)

def init_hmm_from_returns(r, n_states=3):
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    mu0 = np.quantile(r, [0.15, 0.50, 0.85]) if len(r) else np.array([-0.001, 0.0, 0.001])
    var0 = np.full(n_states, max(np.var(r), 1e-6) if len(r) else 1e-6)
    pi0 = np.full(n_states, 1.0 / n_states)
    A0 = np.full((n_states, n_states), 0.05 / (n_states - 1))
    np.fill_diagonal(A0, 0.95)
    return pi0, normalize_rows(A0), mu0, var0

def forward_backward_gaussian(y, pi, A, mu, var):
    y = np.asarray(y, dtype=float)
    T = len(y)
    K = len(mu)
    logB = np.stack([gaussian_logpdf(y, mu[k], var[k]) for k in range(K)], axis=1)

    logA = np.log(np.clip(A, 1e-300, None))
    logpi = np.log(np.clip(pi, 1e-300, None))

    log_alpha = np.empty((T, K))
    log_alpha[0] = logpi + logB[0]
    for t in range(1, T):
        tmp = log_alpha[t - 1][:, None] + logA
        log_alpha[t] = logB[t] + logsumexp(tmp, axis=0)

    log_beta = np.zeros((T, K))
    for t in range(T - 2, -1, -1):
        tmp = logA + logB[t + 1][None, :] + log_beta[t + 1][None, :]
        log_beta[t] = logsumexp(tmp, axis=1)

    loglik = logsumexp(log_alpha[-1], axis=0)
    gamma = np.exp(log_alpha + log_beta - loglik)

    xi = np.zeros((max(T - 1, 0), K, K))
    for t in range(T - 1):
        tmp = log_alpha[t][:, None] + logA + logB[t + 1][None, :] + log_beta[t + 1][None, :] - loglik
        xi[t] = np.exp(tmp)
        s = xi[t].sum()
        if s > 0:
            xi[t] /= s

    return gamma, xi, loglik, log_alpha, log_beta

def baum_welch_gaussian(y, n_states=3, n_iter=5):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) < n_states + 5:
        return init_hmm_from_returns(y, n_states)
    pi, A, mu, var = init_hmm_from_returns(y, n_states)
    for _ in range(n_iter):
        gamma, xi, loglik, *_ = forward_backward_gaussian(y, pi, A, mu, var)
        pi = gamma[0]
        if len(xi) > 0:
            A = normalize_rows(np.clip(xi.sum(axis=0), 1e-12, None))
        w = gamma.sum(axis=0)
        mu = (gamma * y[:, None]).sum(axis=0) / np.maximum(w, 1e-12)
        var = (gamma * (y[:, None] - mu) ** 2).sum(axis=0) / np.maximum(w, 1e-12)
        var = np.clip(var, 1e-8, None)
    return pi, A, mu, var

def viterbi_gaussian(y, pi, A, mu, var):
    y = np.asarray(y, dtype=float)
    T = len(y)
    K = len(mu)
    logB = np.stack([gaussian_logpdf(y, mu[k], var[k]) for k in range(K)], axis=1)
    logA = np.log(np.clip(A, 1e-300, None))
    logpi = np.log(np.clip(pi, 1e-300, None))
    delta = np.empty((T, K))
    psi = np.zeros((T, K), dtype=int)

    delta[0] = logpi + logB[0]
    for t in range(1, T):
        tmp = delta[t - 1][:, None] + logA
        psi[t] = np.argmax(tmp, axis=0)
        delta[t] = np.max(tmp, axis=0) + logB[t]
    states = np.zeros(T, dtype=int)
    states[-1] = np.argmax(delta[-1])
    for t in range(T - 2, -1, -1):
        states[t] = psi[t + 1, states[t + 1]]
    return states

## 8. Fit HMMs on rolling returns

I use a 3-state model on each product's log-return series. The output is:
- transition matrix `A`,
- initial distribution `pi`,
- state means `mu`,
- state variances `var`,
- Viterbi state path,
- filtered regime probabilities.

In [18]:
def fit_product_hmm(df, product, window=None, n_iter=5):
    g = df[df[PRICE_COLS["product"]] == product].sort_values([PRICE_COLS["day"], PRICE_COLS["timestamp"]]).copy()
    r = g["ret"].dropna().astype(float).values
    if window is not None and len(r) > window:
        r = r[-window:]
    if len(r) < 20:
        return None
    pi, A, mu, var = baum_welch_gaussian(r, n_states=3, n_iter=n_iter)
    gamma, xi, loglik, *_ = forward_backward_gaussian(r, pi, A, mu, var)
    states = viterbi_gaussian(r, pi, A, mu, var)
    order = np.argsort(mu)
    return {
        "pi": pi.tolist(),
        "A": A.tolist(),
        "mu": mu.tolist(),
        "var": var.tolist(),
        "ordered_states": order.tolist(),
        "loglik": float(loglik),
        "occ": gamma.mean(axis=0).tolist(),
        "states": states.tolist(),
        "ret_mean": float(np.mean(r)),
        "ret_std": float(np.std(r)),
    }

hmm_results = {}
for p in ALL_PRODUCTS:
    res = fit_product_hmm(prices_f, p, window=3000, n_iter=4)
    if res is not None:
        hmm_results[p] = res

len(hmm_results)

50

In [19]:
# Compact view of the fitted state means and occupancy
rows = []
for p, res in hmm_results.items():
    mu = np.array(res["mu"])
    occ = np.array(res["occ"])
    rows.append({
        "product": p,
        "mu_sorted": tuple(np.round(np.sort(mu), 6)),
        "occ_sorted": tuple(np.round(occ[np.argsort(mu)], 4)),
        "ret_std": res["ret_std"],
        "loglik": res["loglik"],
    })
hmm_table = pd.DataFrame(rows).sort_values("product")
display(hmm_table.head(25))

,product,mu_sorted,occ_sorted,ret_std,loglik
1,GALAXY_SOUNDS_BLACK_HOLES,"(-0.000681, 2.8e-05, 0.000627)","(0.0233, 0.9486, 0.028)",0.000990,16496.732810
0,GALAXY_SOUNDS_DARK_MATTER,"(-0.000698, 6e-06, 0.000517)","(0.0138, 0.9693, 0.0169)",0.001017,16413.969105
2,GALAXY_SOUNDS_PLANETARY_RINGS,"(-0.00061, -2.7e-05, 0.000612)","(0.0294, 0.95, 0.0206)",0.001002,16460.208296
4,GALAXY_SOUNDS_SOLAR_FLAMES,"(-0.000866, -8e-06, 0.000567)","(0.0218, 0.959, 0.0193)",0.001013,16430.027700
3,GALAXY_SOUNDS_SOLAR_WINDS,"(-0.000611, -1.6e-05, 0.000496)","(0.0546, 0.9265, 0.0189)",0.001005,16453.431246
10,MICROCHIP_CIRCLE,"(-0.000611, 5e-06, 0.000722)","(0.021, 0.9583, 0.0208)",0.001026,16390.095309
11,MICROCHIP_OVAL,"(-0.000892, -9e-06, 0.0009)","(0.028, 0.9476, 0.0245)",0.001448,15357.017062
13,MICROCHIP_RECTANGLE,"(-0.000861, 2.5e-05, 0.000874)","(0.0186, 0.9588, 0.0226)",0.001511,15228.191676
12,MICROCHIP_SQUARE,"(-0.0009, -1.6e-05, 0.001015)","(0.0215, 0.9618, 0.0167)",0.001488,15274.865096
14,MICROCHIP_TRIANGLE,"(-0.001009, -1.6e-05, 0.000881)","(0.0254, 0.9481, 0.0265)",0.001507,15235.783581


## 9. Regime classification

The HMM state labels are remapped by mean return:
- lowest mean return → bear,
- middle mean return → neutral,
- highest mean return → bull.

This is the economic ordering step from the HMM theory note. It makes the regimes interpretable and lets the trader switch between trend-following and market-making modes. fileciteturn5file0

In [ ]:
def classify_hmm(res):
    mu = np.array(res["mu"])
    order = np.argsort(mu)
    labels = np.empty(order.shape, dtype=object)
    labels[order] = np.array(["bear", "neutral", "bull"], dtype=object)
    return labels, order

classified = []
for p, res in hmm_results.items():
    labels, order = classify_hmm(res)
    classified.append({
        "product": p,
        "bear_mu": np.array(res["mu"])[order[0]],
        "neutral_mu": np.array(res["mu"])[order[1]],
        "bull_mu": np.array(res["mu"])[order[2]],
        "bear_occ": np.array(res["occ"])[order[0]],
        "neutral_occ": np.array(res["occ"])[order[1]],
        "bull_occ": np.array(res["occ"])[order[2]],
        "avg_duration_bear": 1.0 / max(1e-12, 1.0 - np.array(res["A"])[order[0], order[0]]),
        "avg_duration_neutral": 1.0 / max(1e-12, 1.0 - np.array(res["A"])[order[1], order[1]]),
        "avg_duration_bull": 1.0 / max(1e-12, 1.0 - np.array(res["A"])[order[2], order[2]]),
    })

class_df = pd.DataFrame(classified).sort_values(["bull_occ", "bear_occ"], ascending=False)
display(class_df.head(20))

## 10. Pair spread diagnostics

This cell estimates:
- hedge ratio,
- spread z-score,
- spread half-life proxy,
- entry attractiveness.

The best pair candidates are the ones with the strongest absolute correlation and a spread that actually mean-reverts.

In [ ]:
def spread_stats_for_pair(df, a, b):
    p = df.pivot_table(index=[PRICE_COLS["day"], PRICE_COLS["timestamp"]], columns=PRICE_COLS["product"], values="mid", aggfunc="last").sort_index()
    s = p[[a, b]].dropna()
    if len(s) < 100:
        return None
    x = s[a].values
    y = s[b].values
    beta = np.cov(x, y, ddof=1)[0, 1] / np.var(y, ddof=1) if np.var(y, ddof=1) > 0 else np.nan
    alpha = x.mean() - beta * y.mean()
    spread = x - (alpha + beta * y)
    spread = pd.Series(spread, index=s.index)
    z = (spread - spread.rolling(200, min_periods=50).mean()) / spread.rolling(200, min_periods=50).std()
    ar1 = spread.autocorr(1)
    half_life = np.nan
    if pd.notna(ar1) and 0 < ar1 < 1:
        half_life = -math.log(2) / math.log(ar1)
    return {
        "a": a, "b": b,
        "corr": s[a].corr(s[b]),
        "beta": beta,
        "alpha": alpha,
        "spread_mean": spread.mean(),
        "spread_std": spread.std(),
        "spread_ar1": ar1,
        "half_life": half_life,
        "latest_z": z.dropna().iloc[-1] if len(z.dropna()) else np.nan,
    }

pair_diag = []
for _, row in pairs_df.head(60).iterrows():
    st = spread_stats_for_pair(prices_f, row["a"], row["b"])
    if st:
        pair_diag.append(st)

pair_diag_df = pd.DataFrame(pair_diag).sort_values(["corr", "spread_std"], ascending=False)
display(pair_diag_df.head(25))

## 11. Trend / mean-reversion bucket assignment

This is the final analysis layer. Products are scored into:
- **trend**: strong directional drift plus persistent HMM outer states,
- **mean-revert**: neutral-state dominance and small residual variance,
- **pair-heavy**: strong cross-product spread behaviour,
- **mixed**: no dominant mode, use adaptive market making.

In [ ]:
def bucket_product(row, hmm_row=None):
    if hmm_row is None:
        return "mixed"
    slope = row["slope"]
    cv = row["cv_mid"]
    ret_std = row["ret_std"]
    bull = hmm_row["bull_occ"]
    bear = hmm_row["bear_occ"]
    neutral = hmm_row["neutral_occ"]

    if abs(slope) > 0.05 and (bull > 0.35 or bear > 0.35):
        return "trend"
    if cv < 0.03 and neutral > 0.40:
        return "mean_revert"
    return "mixed"

hmm_lookup = {r["product"]: r for r in classified}
bucket_rows = []
for _, row in prod_summary.iterrows():
    p = row["product"]
    hmm_row = next((r for r in classified if r["product"] == p), None)
    bucket_rows.append({
        "product": p,
        "bucket": bucket_product(row, hmm_row),
        "slope": row["slope"],
        "cv_mid": row["cv_mid"],
        "ret_std": row["ret_std"],
    })
bucket_df = pd.DataFrame(bucket_rows)
display(bucket_df["bucket"].value_counts())
display(bucket_df.head(25))

## 12. Export notebook outputs

The trader should not hardcode arbitrary opinions. It should load:
- the basket map,
- pair candidates,
- HMM parameters,
- per-product bucket labels.

This cell exports those into compact JSON files.

In [ ]:
export = {
    "groups": PRODUCT_GROUPS,
    "product_summary": prod_summary.to_dict(orient="records"),
    "bucket_df": bucket_df.to_dict(orient="records"),
    "pair_diag": pair_diag_df.head(50).to_dict(orient="records"),
    "hmm_results": hmm_results,
}

out_path = DATA_DIR / "round5_analysis_snapshot.json"
with out_path.open("w") as f:
    json.dump(export, f, separators=(",", ":"))

out_path

## 13. Optional backtest scaffold

This final cell is intentionally simple. It can be extended into a real local simulation by replaying historical books and applying the same entry/exit rules as the trader.

In [ ]:
# Example: products with the strongest bull occupancy under HMM
display(class_df.sort_values("bull_occ", ascending=False).head(15)[[
    "product", "bull_occ", "neutral_occ", "bear_occ", "avg_duration_bull", "avg_duration_bear"
]])